# 03 · Positive-unlabeled supervision (B) and geometry-aware decoder (C)

Order (each run is ~6 GPU units of ~8-10 min: 3 CV folds + 3 discovery splits; every unit is resumable):

1. **B0** `b0_reference` - the gated dual encoder re-run with the Phase-5 trainer: the control for everything below.
2. **B1** label-free background weighting · **B2** nnPU prior sweep (0.005 / 0.01 / 0.02).
3. **B3** consistency - only if B1 or B2 passes its promotion gate.
4. **C1** proximity + orientation heads · **C2** + thinness / along-strike continuity · **C3** + soft-clDice.

Set `runtime.max_stage_minutes` in `configs/colab/runtime.yaml` (or per experiment) on short sessions: training stops cleanly at an epoch boundary, checkpoints to Drive, and resumes here.

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'OWNER/REPOSITORY'
COMMIT = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
colab.stage_core(); colab.stage_legacy();

In [ ]:
from gems_bench.experiment import Experiment
C = '/content/GEMS/configs/experiments/'

def run(name):
    e = Experiment.from_yaml(C + name + '.yaml')
    e.run_all()
    d = e.rd.load_table('promotion_decisions') if e.rd.has_table('promotion_decisions') else None
    display(e.status())
    if d is not None:
        display(d)
    return e

b0 = run('b0_reference')

In [ ]:
b1 = run('b1_bg_weight')
b2 = {p: run(f'b2_nnpu_p{p}') for p in ('010', '005', '020')}

In [ ]:
# PU safeguards: confidence mass away from mapped faults and the collapse monitor, vs B0 at the same budget
for e in [b1] + list(b2.values()):
    if e.rd.has_table('pu_confidence_mass'):
        display(e.rd.load_table('pu_confidence_mass'))
    if e.rd.has_table('pu_training_collapse'):
        display(e.rd.load_table('pu_training_collapse'))

In [ ]:
# B3 only if B1 or B2 passed its gate (edit b3_consistency.yaml: pu = winning variant, control = that model)
passed = [e.id for e in [b1] + list(b2.values()) if e.manifest.stage('evaluate').get('outputs', {}).get('promote_recommended')]
print('passed:', passed)
if passed:
    b3 = run('b3_consistency')

In [ ]:
c1 = run('c1_geometry_heads')
c2 = run('c2_thin_continuity')
c3 = run('c3_cldice')
for e in (c1, c2, c3):
    display(e.rd.load_table('geometry_metrics'))
    if e.rd.has_table('orientation_error'):
        display(e.rd.load_table('orientation_error'))